### List of Libraries/Dependencies:
Requires ~40s to import all libraries.

In [ ]:
# Document Converter
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import re
import unicodedata
import json
import logging
import time
from collections.abc import Iterable
from pathlib import Path

from docling_core.types.doc.base import ImageRefMode
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.document import ConversionResult
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption, HTMLFormatOption

# Schema Loader
# Cite the libraryyyyyyyyyyyyyyyyyyyyyyyyyyyyyy
from rdflib import Graph, RDF, RDFS, OWL, URIRef
import rdflib
import os

# Hybrid Chunker
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY (DOCLING)
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY (HUGGINGFACE)
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from transformers import AutoTokenizer
from docling.document_converter import DocumentConverter

# Chunk Processor
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import spacy
from spacy.symbols import VERB, AUX

# Triple Extractor
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import openai
import os

# Triple Extractor - Batch Processing
from concurrent.futures import ThreadPoolExecutor
import json
import time

# Triple Sanitizer
import json

# E-R Normalizer
# 1. MinHash LSH
from datasketch import MinHash, MinHashLSH

# E-R Normalizer
# 2. FAISS Indexing
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Graph Generator
import copy
from rdflib import URIRef, Namespace, RDF, RDFS, OWL, Dataset

# Graph Visualizer
from pyvis.network import Network
import streamlit.components.v1 as components

2026-04-22 00:06:16,189 - INFO - Loading faiss with AVX2 support.
2026-04-22 00:06:16,331 - INFO - Successfully loaded faiss with AVX2 support.


### Document Converter

In [2]:
# Cleans documents
def clean_text(text: str) -> str:
    if not text:
        return text

    # 1. Normalize Unicode (fixes odd composed characters)
    text = unicodedata.normalize("NFKC", text)

    # 2. Remove null bytes (very common in broken conversions)
    text = text.replace("\x00", "")

    # 3. Remove Unicode replacement char (�)
    text = text.replace("\ufffd", "")

    # 4. Remove control characters (but keep newlines/tabs)
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", text)

    return text

_log = logging.getLogger(__name__)

# Export toggles:
# - USE_V2 controls modern Docling document exports.
# - USE_LEGACY enables legacy Deep Search exports for comparison or migration.
USE_V2 = True
USE_LEGACY = False


def export_documents(
    conv_results: Iterable[ConversionResult],
    output_dir: Path,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    failure_count = 0
    partial_success_count = 0

    for conv_res in conv_results:
        if conv_res.status == ConversionStatus.SUCCESS:
            success_count += 1
            doc_filename = conv_res.input.file.stem

            if USE_V2:
                # Export converted files as markdown files
                conv_res.document.save_as_markdown(
                    output_dir / f"{doc_filename}.md",
                    image_mode=ImageRefMode.PLACEHOLDER,
                )
                
                # conv_res.document.save_as_json(
                #     output_dir / f"{doc_filename}.json",
                #     image_mode=ImageRefMode.PLACEHOLDER,
                # )
                
                # conv_res.document.save_as_markdown(
                #     output_dir / f"{doc_filename}.txt",
                #     image_mode=ImageRefMode.PLACEHOLDER,
                #     strict_text=True,
                # )

                # Export Docling document format to markdown:
                with (output_dir / f"{doc_filename}.md").open("w", encoding="utf-8") as fp:
                    raw_md = conv_res.document.export_to_markdown()
                    fp.write(clean_text(raw_md))

                # # Export Docling document format to text:
                # with (output_dir / f"{doc_filename}.txt").open("w") as fp:
                #     fp.write(conv_res.document.export_to_markdown(strict_text=True))

            if USE_LEGACY:
                # Export Markdown format:
                with (output_dir / f"{doc_filename}.legacy.md").open("w", encoding="utf-8") as fp:
                    raw_md = conv_res.document.export_to_markdown()
                    fp.write(clean_text(raw_md))

                # # Export Deep Search document JSON format:
                # with (output_dir / f"{doc_filename}.legacy.json").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(json.dumps(conv_res.document.export_to_dict()))

                # # Export Text format:
                # with (output_dir / f"{doc_filename}.legacy.txt").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(
                #         conv_res.document.export_to_markdown(strict_text=True)
                #     )

                # # Export Document Tags format:
                # with (output_dir / f"{doc_filename}.legacy.doctags.txt").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(conv_res.document.export_to_doctags())

        elif conv_res.status == ConversionStatus.PARTIAL_SUCCESS:
            _log.info(
                f"Document {conv_res.input.file} was partially converted with the following errors:"
            )
            for item in conv_res.errors:
                _log.info(f"\t{item.error_message}")
            partial_success_count += 1
        else:
            _log.info(f"Document {conv_res.input.file} failed to convert.")
            failure_count += 1

    _log.info(
        f"Processed {success_count + partial_success_count + failure_count} docs, "
        f"of which {failure_count} failed "
        f"and {partial_success_count} were partially converted."
    )
    return success_count, partial_success_count, failure_count


def main():
    logging.basicConfig(level=logging.INFO)

    # Location of source documents
    data_folder = Path("./eu_legislation")
    input_doc_paths = [file_path for file_path in data_folder.iterdir()]

    # buf = BytesIO((data_folder / "pdf/2206.01062.pdf").open("rb").read())
    # docs = [DocumentStream(name="my_doc.pdf", stream=buf)]
    # input = DocumentConversionInput.from_streams(docs)

    # # Turn on inline debug visualizations:
    # settings.debug.visualize_layout = True
    # settings.debug.visualize_ocr = True
    # settings.debug.visualize_tables = True
    # settings.debug.visualize_cells = True

    # Configure the PDF pipeline. Enabling page image generation improves HTML
    # previews (embedded images) but adds processing time.
    pdf_pipeline_options = PdfPipelineOptions()
    pdf_pipeline_options.generate_page_images = True

    doc_converter = DocumentConverter(
        allowed_formats=[
            InputFormat.PDF,
            InputFormat.HTML
        ],
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pdf_pipeline_options,
                backend=DoclingParseV4DocumentBackend
            ),
            InputFormat.HTML: HTMLFormatOption()
        }
    )

    start_time = time.time()

    # Convert all inputs. Set `raises_on_error=False` to keep processing other
    # files even if one fails; errors are summarized after the run.
    conv_results = doc_converter.convert_all(
        input_doc_paths,
        raises_on_error=False,  # to let conversion run through all and examine results at the end
    )
    # Write outputs to ./scratch and log a summary.
    _success_count, _partial_success_count, failure_count = export_documents(
        conv_results, output_dir=Path("converted_docs")
    )

    end_time = time.time() - start_time

    _log.info(f"Document conversion complete in {end_time:.2f} seconds.")

    if failure_count > 0:
        raise RuntimeError(
            f"The example failed converting {failure_count} on {len(input_doc_paths)}."
        )

if __name__ == "__main__":
    main()

2026-04-22 00:06:54,762 - INFO - detected formats: [<InputFormat.HTML: 'html'>]


2026-04-22 00:06:54,776 - INFO - Going to convert document batch...
2026-04-22 00:06:54,787 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-04-22 00:06:54,857 - INFO - Loading plugin 'docling_defaults'
2026-04-22 00:06:54,864 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-04-22 00:06:54,865 - INFO - Processing document 31953D0030en.html
2026-04-22 00:06:54,873 - INFO - Finished converting document 31953D0030en.html in 0.11 sec.
2026-04-22 00:06:54,879 - INFO - detected formats: [<InputFormat.HTML: 'html'>]
2026-04-22 00:06:54,889 - INFO - Going to convert document batch...
2026-04-22 00:06:54,889 - INFO - Processing document 31954S0024en.html
2026-04-22 00:06:54,889 - INFO - Finished converting document 31954S0024en.html in 0.02 sec.
2026-04-22 00:06:54,889 - INFO - detected formats: [<InputFormat.HTML: 'html'>]
2026-04-22 00:06:54,907 - INFO - Going to convert document batch...
2026-04-22 00:06:54,907 - INFO - Pr

### Schema Loader

In [3]:
schema_folder = "./schemas"
schemas = [os.path.join(schema_folder, file) for file in os.listdir(schema_folder)]

schemas

['./schemas\\cdm.rdf',
 './schemas\\cdmplus.rdf',
 './schemas\\cdm_annotationproperties.rdf',
 './schemas\\cdm_cataloguing.rdf',
 './schemas\\cdm_datatypes.rdf',
 './schemas\\cdm_indexation.rdf',
 './schemas\\cdm_marc21.rdf',
 './schemas\\euvoc.rdf']

In [4]:
schema_graph = Graph()

index = 1
for file in schemas:
    schema_graph.parse(file, format="xml")
    print(f"Finished parsing RDF file {index}.")
    index += 1

print()

# Only retrieves declared namespaces
schema_namespaces = set(schema_graph.namespaces())
schema_namespaces.remove(('', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdmplus#')))
schema_namespaces.add(('cdmplus#', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdmplus#')))

print("Schema Namespaces:")
for ns in schema_namespaces:
    print(ns)

print()

# Extract classes
classes = set()
for cls in schema_graph.subjects(RDF.type, OWL.Class):
    if not isinstance(cls, rdflib.term.BNode):
        classes.add(cls)

print(f"Found {len(classes)} classes.")

# Extract object properties
obj_properties = set()
for prop in schema_graph.subjects(RDF.type, OWL.ObjectProperty):
    if not isinstance(prop, rdflib.term.BNode):
        obj_properties.add(prop)

print(f"Found {len(obj_properties)} object properties.")

# Extract datatype properties
datatype_properties = set()
for prop in schema_graph.subjects(RDF.type, OWL.DatatypeProperty):
    if not isinstance(prop, rdflib.term.BNode):
        datatype_properties.add(prop)

print(f"Found {len(datatype_properties)} datatype properties.")

# Extract domain and range for all properties
prop_domains = dict()
prop_ranges = dict()

for prop in obj_properties.union(datatype_properties):
    domains = set(schema_graph.objects(prop, RDFS.domain))
    ranges = set(schema_graph.objects(prop, RDFS.range))
    prop_domains[prop] = domains
    prop_ranges[prop] = ranges

# Pretty-print helper function
def pretty_uri(uri):
    if isinstance(uri, URIRef):
        return str(uri).split('#')[-1]
    return str(uri)

print("\nSample property info:")
for prop in list(prop_domains.keys())[:5]:
    print(f"{pretty_uri(prop)}:")
    print(f"\tDomains:\t{[pretty_uri(d) for d in prop_domains[prop]]}")
    print(f"\tRanges:\t{[pretty_uri(r) for r in prop_ranges[prop]]}")


Finished parsing RDF file 1.
Finished parsing RDF file 2.
Finished parsing RDF file 3.
Finished parsing RDF file 4.
Finished parsing RDF file 5.
Finished parsing RDF file 6.
Finished parsing RDF file 7.
Finished parsing RDF file 8.

Schema Namespaces:
('admin', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdm/admin#'))
('org', rdflib.term.URIRef('http://www.w3.org/ns/org#'))
('geo1', rdflib.term.URIRef('http://www.w3.org/2003/01/geo/wgs84_pos#'))
('sh', rdflib.term.URIRef('http://www.w3.org/ns/shacl#'))
('ssn', rdflib.term.URIRef('http://www.w3.org/ns/ssn/'))
('skosxl', rdflib.term.URIRef('http://www.w3.org/2008/05/skos-xl#'))
('prof', rdflib.term.URIRef('http://www.w3.org/ns/dx/prof/'))
('rdfs', rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#'))
('cdmplus#', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdmplus#'))
('odrl', rdflib.term.URIRef('http://www.w3.org/ns/odrl/2/'))
('sosa', rdflib.term.URIRef('http://www.w3.org/ns/sosa/'))
('owl', rdflib.

#### Resource-URI dictionaries

In [5]:
classes_dict = {}
class_uris = list(classes)
class_names = [str(cls).rsplit("/", 1)[1] for cls in class_uris]
for name, uri in zip(class_names, class_uris):
    classes_dict[name] = uri

obj_prop_dict = {}
obj_prop_uris = list(obj_properties)
obj_prop_names = [str(obj_prop).rsplit("/", 1)[1] for obj_prop in obj_prop_uris]
for name, uri in zip(obj_prop_names, obj_prop_uris):
    obj_prop_dict[name] = uri

data_prop_dict = {}
data_prop_uris = list(datatype_properties)
data_prop_names = [str(data_prop).rsplit("/", 1)[1] for data_prop in data_prop_uris]
for name, uri in zip(data_prop_names, data_prop_uris):
    data_prop_dict[name] = uri

classes_str = ", ".join(classes_dict.keys())
obj_prop_str = ", ".join(obj_prop_dict.keys())
data_prop_str = ", ".join(data_prop_dict.keys())

rdf_resources = list(classes_dict.keys()) + list(obj_prop_dict.keys()) + list(data_prop_dict.keys())
rdf_dict = {**classes_dict, **obj_prop_dict, **data_prop_dict}

##### *Inspecting the Retrieved Classes:*

In [6]:
list(classes_dict.keys())

['cdm#collection_document',
 'cdm#ATTO_FD_501',
 'cdmplus#CaseLawWork',
 'cdmplus#Agent',
 'cdm#summary-executive-study-evaluation',
 'cdm#initiative-priority-ec',
 'cdm#cooperation_police-and-judicial',
 'cdmplus#EFTAInternationalAgreementItem',
 'cdm#act_preparatory_ecsc',
 'cdm#recording-video',
 'cdmplus#ThirdPartyProceedingItem',
 'cdm#manifestation-publication-general',
 'Standard',
 'cdm#question_oral',
 'cdm#agreement_international',
 'cdm#opinion_ecb',
 'cdm#place',
 'cdm#agreement_international_efta',
 'cdm#event_case',
 'cdm#case-law_national',
 'cdm#sec_package_ec',
 'cdmplus#Communication',
 'euvoc#DatasetTheme',
 'cdm#expression',
 'cdmplus#PreparatoryActExpression',
 'cdm#proposal_regulation_codified_ec',
 'cdm#ATTO_FD_610',
 'cdmplus#ComFinalWork',
 'cdmplus#MergersAndStateAidAndConcentrationsWork',
 'cdm#summary',
 'cdmplus#SecondaryLawExpression',
 'Location',
 'cdm#session-eesc-committee',
 'cdmplus#NonOppositionToNotifiedJointVentureExpression',
 'cdm#concept_field_

In [7]:
print("No. of RDF Resources:", len(rdf_resources))

No. of RDF Resources: 5316


### Hybrid Chunker

In [8]:
EMBED_MODEL_ID = "openai/gpt-oss-120b"

MAX_TOKENS = 1024

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
    max_tokens=MAX_TOKENS,
)

chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,
)

digitalizer = DocumentConverter()

# CHANGE THIS
conv_docs_folder = Path("./converted_docs")
conv_doc_paths = [file_path for file_path in conv_docs_folder.iterdir()]

chunk_batches = list()

for doc_path in conv_doc_paths:
    filename = os.path.basename(doc_path)
    doc = digitalizer.convert(source=doc_path).document
    chunk_iter = chunker.chunk(dl_doc=doc)
    current_chunks = list(chunk_iter)
    chunk_batches.append({"chunks": current_chunks, "filename": filename})

2026-04-22 00:12:53,148 - INFO - detected formats: [<InputFormat.MD: 'md'>]
2026-04-22 00:12:53,150 - INFO - Going to convert document batch...
2026-04-22 00:12:53,151 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-04-22 00:12:53,152 - INFO - Processing document 31953D0030en.md
2026-04-22 00:12:53,185 - INFO - Finished converting document 31953D0030en.md in 0.05 sec.
2026-04-22 00:12:53,372 - INFO - detected formats: [<InputFormat.MD: 'md'>]
2026-04-22 00:12:53,372 - INFO - Going to convert document batch...
2026-04-22 00:12:53,372 - INFO - Processing document 31954S0024en.md
2026-04-22 00:12:53,409 - INFO - Finished converting document 31954S0024en.md in 0.03 sec.
2026-04-22 00:12:53,410 - INFO - detected formats: [<InputFormat.MD: 'md'>]
2026-04-22 00:12:53,421 - INFO - Going to convert document batch...
2026-04-22 00:12:53,422 - INFO - Processing document 31954S0026en.md
2026-04-22 00:12:53,448 - INFO - Finished converting d

##### *Chunk Inspection:*

In [9]:
sample_chunk_batch = chunk_batches[0]
sample_chunk_filename = sample_chunk_batch["filename"]
sample_chunk_list = sample_chunk_batch["chunks"]

print("The current chunks are from:", sample_chunk_filename)
for i, chunk in enumerate(sample_chunk_list):
    print(f"=== {i} ===")
    txt_tokens = tokenizer.count_tokens(chunk.text)
    print(f"chunk.text ({txt_tokens} tokens):\n{chunk.text!r}")

    ser_txt = chunker.contextualize(chunk=chunk)
    ser_tokens = tokenizer.count_tokens(ser_txt)
    print(f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_txt!r}")

    print()    

The current chunks are from: 31953D0030en.md
=== 0 ===
chunk.text (303 tokens):
"**ECSC High Authority: Decision No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel**\n*Official Journal 006 , 04/05/1953 P. 0109 - 0110 Danish special edition: Series I Chapter 1952-1958 P. 0009 English special edition: Series I Chapter 1952-1958 P. 0009 Greek special edition: Chapter 08 Volume 1 P. 0005 Spanish special edition: Chapter 08 Volume 1 P. 0005 Portuguese special edition Chapter 08 Volume 1 P. 0005 Finnish special edition: Chapter 12 Volume 3 P. 0003 Swedish special edition: Chapter 12 Volume 3 P. 0003*\nDECISION No 30-53  of 2 May 1953  on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel THE HIGH AUTHORITY, Having regard to Article 60 and Article 63 (2) of the Treaty; Whereas compliance with the obligations of non-discrimination involves uniform application by undertakings of the con

### Chunk Processor

In [10]:
nlp = spacy.load("en_core_web_trf")

processed_chunks = []

def strip_all_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

for batch in chunk_batches:
    filename = batch["filename"]
    chunk_list = batch["chunks"]

    for chunk in chunk_list:
        doc = nlp(chunk.text)

        filtered_sents = []

        for sent in doc.sents:
            text = sent.text.strip()

            if not text:
                continue

            sent_doc = sent.as_doc()

            # check for VERB or AUX in the sentence
            has_verb_or_aux = any(
                token.pos_ in {"VERB", "AUX"} for token in sent_doc
            )

            if not has_verb_or_aux:
                continue

            filtered_sents.append(text)

        if not filtered_sents:
            continue

        processed_chunk = " ".join(filtered_sents)

        cleaned = strip_all_whitespace(processed_chunk)

        if cleaned:  # simpler + safer than comparing twice
            processed_chunks.append({
                "content": cleaned,
                "filename": filename
            })

        print("CHUNK:", repr(cleaned))
        print("FILE:", filename, "\n")

CHUNK: "**ECSC High Authority: Decision No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel** *Official Journal 006 , 04/05/1953 P. 0109 - 0110 Danish special edition: Series I Chapter 1952-1958 P. 0009 English special edition: Series I Chapter 1952-1958 P. 0009 Greek special edition: Chapter 08 Volume 1 P. 0005 Spanish special edition: Chapter 08 Volume 1 P. 0005 Portuguese special edition Chapter 08 Volume 1 P. 0005 Finnish special edition: Chapter 12 Volume 3 P. 0003 Swedish special edition: Chapter 12 Volume 3 P. 0003* DECISION No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel THE HIGH AUTHORITY, Having regard to Article 60 and Article 63 (2) of the Treaty; Whereas compliance with the obligations of non-discrimination involves uniform application by undertakings of the conditions shown in their price lists with no other increases or reductions and 

In [11]:
len(processed_chunks)

231

### Triple Extractor

ISSUE: Some triples contain unknown code, e.g \u202f.

PROPOSAL: Check embeddings.

In [12]:
# Security Measure
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.getenv('OPENAI_API_KEY')

# REMINDER: Refresh API key if new instances are needed
client = openai.OpenAI(
    base_url = "https://integrate.api.nvidia.com/v1",
    api_key = openai.api_key
    )

schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "subject": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "minLength": 1
                    },
                    "labels": {
                        "type": "array",
                        "items": {"type": "string"},
                    }
                },
                "required": ["name", "labels"],
                "additionalProperties": False
            },
            "predicate": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                    },
                    "labels": {
                        "type": "array",
                        "items": {"type": "string"},
                    }
                },
                "required": ["name", "labels"],
                "additionalProperties": False
            },
            "object": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "minLength": 1
                    },
                    "labels": {
                        "type": "array",
                        "items": {"type": "string"},
                    }
                },
                "required": ["name", "labels"],
                "additionalProperties": False
            }
        },
        "required": ["subject", "predicate", "object"],
        "additionalProperties": False
    }
}

system_prompt = f"""
Extract semantic triples from the provided text.

You MUST follow the externally enforced JSON schema. Do not describe or assume output formatting.

Task:
- Identify factual subject–predicate–object relations in the text.

Semantic rules:
- Replace pronouns and references with their correct entities.
- Split compound or coordinated entities (e.g., "A and B") into separate triples.
- Use only explicit or clearly implied relations from the text.
- Avoid duplicate or redundant triples.
- Prefer the most specific ontology labels provided.

Labeling rules:
- Assign appropriate labels from the provided ontology:
  - Classes
  - Object Properties
  - Data Properties
- You may directly assign labels as predicates (e.g., rdf:type)
- You may assign multiple labels when appropriate.

Output rules:
- Do NOT output explanations or commentary.
- Do NOT attempt to format JSON manually.
- Output structure is fully controlled by the external schema.

If no valid triples exist, return an empty result as defined by the schema.

Classes: {classes_str}
Object Properties: {obj_prop_str}
Data Properties: {data_prop_str}
"""

def get_completion(system_prompt=system_prompt, query=""):
    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
           {'role':'system', 'content': system_prompt},
           {'role':'user', 'content': query}
           ],
        temperature=0,
        top_p=0.1, # Test different values
        max_tokens=None,
        stream=False,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "triples",
                "schema": schema
                }
            }
        )
    
    reasoning = getattr(completion.choices[0].message, "reasoning_content", None)
    
    #if reasoning:
    #   print(reasoning)

    return completion.choices[0].message.content


#### Batch Processing

In [13]:
# Parse outputs
responses = list()
chunk_filenames = set([chunk["filename"] for chunk in processed_chunks])
triples_by_filename = {filename: list() for filename in chunk_filenames}

def extraction_thread(chunk_dict):
    response = get_completion(
        system_prompt=system_prompt,
        query=chunk_dict["content"]
    )

    return chunk_dict["filename"], response

with ThreadPoolExecutor(max_workers=10) as executor:
    responses = list(executor.map(extraction_thread, processed_chunks))

for filename, item in responses:
    if type(item) == str:
        triples_by_filename[filename].append(item)
    elif type(item) == list:
        triples_by_filename[filename].extend(item)

2026-04-22 00:21:57,386 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 00:22:00,351 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 00:22:02,515 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 00:22:08,049 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 00:22:08,849 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 00:22:08,982 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 00:22:12,225 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 00:22:16,348 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 00:22

### Triple Sanitizer

In [14]:
def safe_parse(triple):
    try:
        return json.loads(triple)
    except json.JSONDecodeError:
        return None

def repair_json(text: str):
    text = text.strip()

    # Remove trailing junk after last valid closing bracket
    match = re.search(r"(\{.*\}|\[.*\])", text)
    if match:
        text = match.group(0)

    # Fix common bracket imbalance
    open_brackets = text.count("[")
    close_brackets = text.count("]")
    if open_brackets > close_brackets:
        text += "]" * (open_brackets - close_brackets)

    open_braces = text.count("{")
    close_braces = text.count("}")
    if open_braces > close_braces:
        text += "}" * (open_braces - close_braces)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

In [15]:
cleaned_data = dict()

for filename, triple_list in triples_by_filename.items():
    parsed_triples = list()

    for triple in triple_list:
        triple = triple.strip()
        if not triple:
            continue

        parsed = safe_parse(triple)

        if parsed is None:
            parsed = repair_json(triple)

        if parsed is None:
            print("FAILED:", triple)
            continue

        parsed_triples.extend(parsed)

    cleaned_data[filename] = parsed_triples

len(cleaned_data)


99

##### *Inspecting the Cleaned Triples:*

In [16]:
total = 0

for triple_list in cleaned_data.values():
    total += len(triple_list)

total

4073

In [22]:
cleaned_data["31958D1127_01_en.md"]

[{'subject': {'name': 'cdm#document-ec',
   'labels': ['Rules of the Transport Committee']},
  'predicate': {'name': 'cdm#adopted_by', 'labels': []},
  'object': {'name': 'cdm#entity_council', 'labels': ['EEC Council']}},
 {'subject': {'name': 'cdm#document-ec', 'labels': []},
  'predicate': {'name': 'cdm#issued_by', 'labels': []},
  'object': {'name': 'cdm#official-journal', 'labels': []}},
 {'subject': {'name': 'cdm#document-ec', 'labels': []},
  'predicate': {'name': 'cdm#has_type', 'labels': []},
  'object': {'name': 'cdm#official-journal-act', 'labels': []}},
 {'subject': {'name': 'cdm#committee', 'labels': ['Transport Committee']},
  'predicate': {'name': 'cdm#has_part', 'labels': []},
  'object': {'name': 'cdm#person', 'labels': ['expert']}},
 {'subject': {'name': 'cdm#organization', 'labels': ['European Commission']},
  'predicate': {'name': 'cdm#consults', 'labels': []},
  'object': {'name': 'cdm#committee', 'labels': ['Transport Committee']}},
 {'subject': {'name': 'cdm#repor

#### *Trimming unnecessary prefixes:*

In [62]:
def strip_cdm_prefix(name: str) -> str:
    name_lower = name.lower()

    # preserve these EXACT prefixes
    if name_lower.startswith("cdm#") or name_lower.startswith("cdmplus"):
        return name

    # remove only raw "cdm" prefix
    if name_lower.startswith("cdm"):
        return name[3:]

    return name

In [73]:
subjs_objs = set()

for filename, triple_list in cleaned_data.items():
    for triple in triple_list:

        # clean subject
        subj = triple["subject"]["name"]
        subj_clean = strip_cdm_prefix(subj)
        triple["subject"]["name"] = subj_clean

        # clean object
        obj = triple["object"]["name"]
        obj_clean = strip_cdm_prefix(obj)
        triple["object"]["name"] = obj_clean

        # always use cleaned values
        subjs_objs.add(subj_clean)
        subjs_objs.add(obj_clean)

subjs_objs = list(subjs_objs)

In [64]:
len(subjs_objs)

3126

### Triple Labelling (CONSIDER REMOVING)

ISSUE: Some entity names can be shortened, but re-integrating them into the triples list is difficult.

In [131]:
labelling_prompt_entities = """
From the given JSON string of a semantic triplet, add in two key-value pairs: "schema-type" and "confidence".
For example:
"subject": {
    "surface_form": "Republic of Austria",
    "role": "entity"
    "schema-type": [type_1, type_2, ... type_n]
    "confidence": 0.8
  }
"""
labelling_prompt_entities += f"Label the schema types based on the following RDF classes and object properties:\n1. Classes: {classes_str};\n2. Object Properties: {obj_prop_str}.\nBe mindful that an entity may have multiple labels.\nIn particular, labels prefixed with #cdm and #euvoc often share names.\nIf there are no suitable matches, write None."

# TEST
label_responses_entities = []
for entity in subjs_objs[:10]: # For testing purposes, REMOVE THE INDEX WHEN DONE.
    label_responses_entities.append(get_completion(system_prompt=labelling_prompt_entities, prompt=str(entity)))

2026-03-02 19:29:29,257 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:29:34,987 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:29:39,802 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:29:42,365 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:29:45,551 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:29:49,017 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:29:51,893 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:29:55,069 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:30

In [132]:
label_responses_entities

['{\n  "surface_form": "Entry into force",\n  "role": "entity",\n  "schema-type": [\n    "cdm#event",\n    "cdm#event_legal"\n  ],\n  "confidence": 0.9\n}',
 '{\n  "surface_form": "Chapter 13 Article 2 (1) and (2) and Chapter 14 Article 1 (3) of sjolagen (maritime law)",\n  "role": "entity",\n  "schema-type": ["cdm#article_legal"],\n  "confidence": 0.9\n}',
 '{\n  "surface_form": "korkein hallinto-oikeus/högsta förvaltningsdomstolen",\n  "role": "entity",\n  "schema-type": ["cdm#court_national"],\n  "confidence": 0.9\n}',
 '{\n  "surface_form": "German",\n  "role": "entity",\n  "schema-type": [\n    "euvoc#Country",\n    "cdm#country"\n  ],\n  "confidence": 0.95\n}',
 '{\n  "surface_form": "Secretary-General of the Council of the European Union",\n  "role": "entity",\n  "schema-type": [\n    "cdm#person",\n    "cdm#agent"\n  ],\n  "confidence": 0.9\n}',
 '{\n  "surface_form": "Spanish",\n  "role": "entity",\n  "schema-type": [\n    "euvoc#Language",\n    "cdm#language"\n  ],\n  "confid

In [133]:
labelling_prompt_relations = """
From the given JSON string of a semantic triplet, add in two key-value pairs: "data-type" and "confidence".
For example:
"predicate": {
    "surface_form": "accessed",
    "role": "relation"
    "data-type": [type_1, type_2, ... type_n]
    "confidence": 0.8
  }
"""
labelling_prompt_relations += f"Label the schema types based on the following RDF data properties:\nData Properties: {data_prop_str}.\nBe mindful that an entity may have multiple labels.\nIn particular, labels prefixed with #cdm and #euvoc often share names.\nIf there are no suitable matches, write None."

# TEST
label_responses_relations = []
for relation in rels[:10]: # For testing purposes, REMOVE THE INDEX WHEN DONE.
    label_responses_relations.append(get_completion(system_prompt=labelling_prompt_relations, prompt=str(relation)))

2026-03-02 19:33:50,068 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:33:53,144 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:33:56,524 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:33:59,472 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:34:03,062 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:34:05,745 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:34:10,579 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:34:13,007 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 19:34

In [134]:
label_responses_relations

['{\n  "surface_form": "accedes to",\n  "role": "relation",\n  "data-type": null,\n  "confidence": 0.8\n}',
 '{\n  "surface_form": "is in",\n  "role": "relation",\n  "data-type": null,\n  "confidence": 0.0\n}',
 '{\n  "surface_form": "may retain",\n  "role": "relation",\n  "data-type": null,\n  "confidence": 0.0\n}',
 '{\n  "surface_form": "occurred on",\n  "role": "relation",\n  "data-type": ["date"],\n  "confidence": 0.9\n}',
 '{\n  "surface_form": "drawn up in",\n  "role": "relation",\n  "data-type": [\n    "cdm#work_date_creation",\n    "cdm#date"\n  ],\n  "confidence": 0.75\n}',
 '{\n  "surface_form": "deposits",\n  "role": "relation",\n  "data-type": null,\n  "confidence": 0.0\n}',
 '{\n  "surface_form": "shall be deposited in",\n  "role": "relation",\n  "data-type": null,\n  "confidence": 0.3\n}',
 '{\n  "surface_form": "become member of",\n  "role": "relation",\n  "data-type": null,\n  "confidence": 0.0\n}',
 '{\n  "surface_form": "transmit to",\n  "role": "relation",\n  "data-

### Entity-Relation Normalizer (INCOMPLETE)

In [43]:
def ngram_tokenize(text, n=3):
    text = text.lower().replace(" ", "")
    return [text[i: i + n] for i in range(len(text) - n + 1)]

In [65]:
# MinHash LSH
num_perm = 256
lsh = MinHashLSH(threshold=0.5, num_perm=num_perm)
minhash_store = {}

# Build index
for idx, entity in enumerate(subjs_objs):
    m = MinHash(num_perm=num_perm)
    for token in ngram_tokenize(entity):
        m.update(token.encode("utf8"))
    key = idx
    lsh.insert(key, m)
    minhash_store[key] = m

In [66]:
# FAISS Indexing
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    subjs_objs,
    normalize_embeddings=True
    ).astype(np.float32)

dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

# Only use the following when candidate sets are small (cand_set < 200)
# scores = np.dot(candidate_vecs, query_vec.T).flatten()

2026-04-22 01:44:36,379 - INFO - Use pytorch device_name: cpu
2026-04-22 01:44:36,381 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Batches:   0%|          | 0/98 [00:00<?, ?it/s]

In [67]:
candidate_map = {}
final_pairs = set()
threshold = 0.9

for i in range(len(subjs_objs)):
    candidates = lsh.query(minhash_store[i])
    candidate_ids = [c for c in candidates if c != i]
    candidate_map[i] = candidate_ids

for i, candidate_ids in candidate_map.items():
    
    if len(candidate_ids) == 0:
        continue
    
    query_vec = embeddings[i].reshape(1, -1)
    candidate_vecs = embeddings[candidate_ids]

    sub_index = faiss.IndexFlatIP(candidate_vecs.shape[1])
    sub_index.add(candidate_vecs)

    distances, indices = sub_index.search(query_vec, k=len(candidate_ids))

    for rank, idx in enumerate(indices[0]):
        score = float(distances[0][rank])
        j = candidate_ids[idx]

        if score >= threshold and i < j:
            final_pairs.add((i, j, score))


In [68]:
len(final_pairs)

738

In [69]:
matched_list = []

for index, neighbor, score in final_pairs:
    if score == 1.0:
        matched_list.append((subjs_objs[index], subjs_objs[neighbor], score))

matched_list

[('court\u202fof\u202fjustice', 'Court of Justice', 1.0),
 ('directorsrepresentingelectricitedefrance',
  'DirectorsRepresentingElectricitéDeFrance',
  1.0),
 ('series\u202fi', 'series i', 1.0),
 ('director of directorate b of the directorate‑general for agriculture',
  'Director of Directorate\u202fB of the Directorate‑General for Agriculture',
  1.0)]

*Naive Deduplication:*

In [70]:
for triple_list in cleaned_data.values():
     sim_list = [
          (id, match[0].lower())
          for id, triple in enumerate(triple_list)
          for match in matched_list
          if (match[0].lower() == (triple["subject"]["name"].lower())) or (match[0].lower() == triple["object"]["name"].lower())
          ]

     for id, reference in sim_list:
          if triple_list[id]["subject"]["name"].lower() == reference:
               triple_list[id]["subject"]["name"] = reference
          if triple_list[id]["object"]["name"].lower() == reference:
               triple_list[id]["object"]["name"] = reference

          print(triple_list[id]["subject"]["name"], triple_list[id]["object"]["name"])

court of justice review body for penalties under Regulation No 11
danish_special_edition series i
english_special_edition series i
directorsrepresentingelectricitedefrance ElectriciteDeFrance
subparagraph (c) of Article 1 (2) director of directorate b of the directorate‑general for agriculture
director of directorate b of the directorate‑general for agriculture cdmplus#Agent


### Schema Validator (FUTURE WORK)

### Graph Serializer

In [100]:
KNOWN_PREDICATES = {
    "rdf:type": RDF.type,
    "type": RDF.type,
    "rdfs:label": RDFS.label,
    "label": RDFS.label,
    "owl:sameAs": OWL.sameAs
}

def normalize_uri(text):
    text = text.strip()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^\w\-\#]", "", text)  # keep #
    return text

def resolve_resource(p):
    # normalize
    key = p.strip()

    # ✅ case 1: known RDF predicate
    for known in KNOWN_PREDICATES.keys():
        if key.lower() == known.lower():
            key = known
            return KNOWN_PREDICATES[key]

    # ✅ case 2: in your ontology dict
    for resource in rdf_dict.keys():
        if key.lower() == resource.lower():
            key = resource
            return URIRef(rdf_dict[key])

    # ✅ case 3: fallback → your namespace
    return JS[key.replace(" ", "_")]

In [102]:
pure_triples = {filename: set() for filename in cleaned_data.keys()}
errors = {filename: list() for filename in cleaned_data.keys()}

JS = Namespace("http://jurisynth.org/cdmext/data/")

for filename, triple_list in copy.deepcopy(cleaned_data).items():
    for triple in triple_list:

        subj = normalize_uri(triple["subject"]["name"])
        obj = normalize_uri(triple["object"]["name"])

        pred_labels = triple["predicate"].get("labels", [])

        if pred_labels:
            for pred_tag in pred_labels:
                try:
                    pred = resolve_resource(pred_tag)

                    pure_triples[filename].add((JS[subj], pred, JS[obj]))

                except Exception:
                    errors[filename].append((JS[subj], pred_tag, JS[obj], 1))

        else:
            key = triple["predicate"]["name"]

            pred = resolve_resource(key)

            pure_triples[filename].add((JS[subj], pred, JS[obj]))

total = sum([len(triple_list) for triple_list in pure_triples.values()])
print("No. of proper triples:", total)

subj_tags = copy.deepcopy([
    (triple['subject']["name"], tuple(triple['subject']["labels"]), filename)
    for filename, triple_list in cleaned_data.items()
    for triple in triple_list
    ])
obj_tags = copy.deepcopy([
    (triple['object']["name"], tuple(triple['object']["labels"]), filename)
    for filename, triple_list in cleaned_data.items()
    for triple in triple_list
    ])
triple_tags = set(subj_tags + obj_tags)

empty_tags = set([pair for pair in triple_tags if len(pair[1]) == 0])
triple_tags -= empty_tags

for entity, label_list, filename in triple_tags:
    norm_entity = normalize_uri(entity)
    for label in label_list:
        clean_label = normalize_uri(strip_cdm_prefix(label))
        resolved_label = resolve_resource(clean_label)

        pure_triples[filename].add((
            JS[norm_entity],
            RDF.type,
            resolved_label
            ))

total = sum([len(triple_list) for triple_list in errors.values()])
print()
print("No. of erroneous triples:", total)

No. of proper triples: 4056

No. of erroneous triples: 0


*Issue (Future Work):*
* Route highly-matched resources to their appropriate namespaces.

In [103]:
pure_triples

{'31961D0408_01_en.md': {(rdflib.term.URIRef('http://jurisynth.org/cdmext/data/1961-03-08'),
   rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
   rdflib.term.URIRef('http://purl.org/dc/terms/date')),
  (rdflib.term.URIRef('http://jurisynth.org/cdmext/data/Commission_Decision_on_modification_of_aid_system_in_Italy'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/addresses'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/compatibility_of_Italian_draft_law_with_common_market_under_Article_923c')),
  (rdflib.term.URIRef('http://jurisynth.org/cdmext/data/Commission_Decision_on_modification_of_aid_system_in_Italy'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/date_adopted'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/1961-03-08')),
  (rdflib.term.URIRef('http://jurisynth.org/cdmext/data/Commission_Decision_on_modification_of_aid_system_in_Italy'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/issued_by'),
   rdfli

In [104]:
errors_list = list()

for err_list in errors.values():
    errors_list.extend(err_list)

len(errors_list)

0

*Handling Erroneous Triples (if any):*

In [140]:
def batch_list(lst, size=10):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

def error_fix_batch(errors):
    batch_prompt = "Fix the labels for the following triples.\n\n"

    metadata = []

    for i, error in enumerate(errors):
        err_index = error[-1]
        incomplete_tag = error[err_index]
        
        if "#" not in incomplete_tag:
            continue  # or handle differently

        reference = incomplete_tag.split("#")[-1]
        triple = ", ".join(error[:3])
        query_list = [key for key in rdf_dict.keys() if reference in key]

        batch_prompt += f"""
        Item {i}:
        Triple: {triple}
        Incorrect label: {incomplete_tag}
        Candidates: {", ".join(query_list)}
        """

        metadata.append((i, error, err_index))

    batch_prompt += """
    Return the corrected labels as a JSON array of strings in order:
    ["label1", "label2", ...]
    Only return the JSON array.
    """

    # 🔑 ONE API CALL
    labels = json.loads(get_completion(system_prompt=batch_prompt, query=""))

    corrected = []

    for label, (_, error, err_index) in zip(labels, metadata):
        triple = list(error[:3])
        try:
            triple[err_index] = rdf_dict[label]
        except:
            print(triple, err_index)
        trp_dict = {"triple": triple, "filename": filename}
        corrected.append(triple)

    return corrected

In [ ]:
error_batches = list(batch_list(errors_list, 10))

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(error_fix_batch, error_batches))

fixed_list = []
for batch in results:
    fixed_list.extend(batch)

In [105]:
named_graph = Dataset()
SOURCE = Namespace("http://jurisynth.org/cdmext/source/")

named_graph.bind("js", JS)
named_graph.bind("source", SOURCE)

for idx, ns in schema_namespaces:
    named_graph.bind(idx, ns)

for filename, triple_list in pure_triples.items():

    graph_uri = SOURCE[normalize_uri(filename)]
    g = named_graph.graph(graph_uri)

    for s, p, o in triple_list:
        g.add((s, p, o))
        
named_graph.serialize("eu_legislation_graph.nq", format="nquads")

<Graph identifier=N65eac2211c334231b44be1f3bfb8745d (<class 'rdflib.graph.Dataset'>)>